# Topological-stability diagnostics for caged states

This notebook implements the first three numerical tests proposed for the topological-stability program:

1. **one-parameter branch tracking**, separating a continued cage eigenstate from the unchanged pre-quench state;
2. **random multi-parameter ensembles**, comparing compatible and incompatible local directions;
3. **the linearized obstruction map**, including both the boundary-cancellation equation and the internal eigenvalue equation.

The notebook begins with a transparent four-state toy cancellation network, then applies the same workflow to the known square-QDM \(4\times4\), \(W=(0,0)\), \((\kappa,Z)=(0,4)\) cage.

The tests establish exact or structural stability inside a specified perturbation class. They do **not** yet establish a topological invariant; that is the next stage of the project.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make the notebook runnable both from the repository root and from this folder.
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "qlinks").is_dir():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the qlinks repository root.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qlinks.builders import SparseHamiltonianBuilder
from qlinks.caging import (
    CageSearchConfig,
    cage_compatibility_hierarchy_from_hamiltonians,
    CageSearcher,
    combine_perturbations_from_coefficients,
    diagnose_cage_stability,
    estimate_power_law_exponent,
    linearized_cage_obstruction_from_hamiltonians,
    random_cage_stability_ensemble,
    scan_cage_stability_branch,
    scan_support_eigenstate_branch,
)
from qlinks.models import SquareQDMModel

## 1. Toy cancellation network

The support contains two Fock-space vertices.  The boundary map

\[
B_0=\begin{pmatrix}1&1\\0&0\end{pmatrix}
\]

annihilates the antisymmetric cage vector \((1,-1)/\sqrt2\).  We compare:

- a **strongly compatible** perturbation that keeps the same vector;
- a **structure-compatible** perturbation that rotates the boundary kernel and therefore deforms the cage vector;
- an **incompatible** perturbation that adds an independent boundary row and removes the kernel.

In [ ]:
def assemble_hamiltonian(boundary, internal=None, external=None):
    if internal is None:
        internal = np.zeros((2, 2), dtype=np.complex128)
    if external is None:
        external = np.diag([2.0, 3.0]).astype(np.complex128)
    return np.block([[internal, boundary.conj().T], [boundary, external]])

base_boundary = np.array([[1.0, 1.0], [0.0, 0.0]], dtype=np.complex128)
base_hamiltonian = assemble_hamiltonian(base_boundary)

strong_perturbation = assemble_hamiltonian(
    np.array([[1.0, 1.0], [0.0, 0.0]], dtype=np.complex128),
    internal=np.eye(2, dtype=np.complex128),
    external=np.zeros((2, 2), dtype=np.complex128),
)
structure_perturbation = assemble_hamiltonian(
    np.array([[0.0, 1.0], [0.0, 0.0]], dtype=np.complex128),
    external=np.zeros((2, 2), dtype=np.complex128),
)
incompatible_perturbation = assemble_hamiltonian(
    np.array([[0.0, 0.0], [1.0, 0.0]], dtype=np.complex128),
    external=np.zeros((2, 2), dtype=np.complex128),
)

support = (0, 1)
cage_state = np.array([1.0, -1.0], dtype=np.complex128) / np.sqrt(2.0)

baseline = diagnose_cage_stability(
    base_hamiltonian,
    support,
    state=cage_state,
    tolerance=1.0e-12,
)
baseline.to_summary_dict()

In [ ]:
parameters = np.linspace(0.0, 1.0, 21)
structure_branch = scan_cage_stability_branch(
    base_hamiltonian,
    structure_perturbation,
    support,
    parameters,
    reference_state=cage_state,
    tolerance=1.0e-12,
)
incompatible_branch = scan_cage_stability_branch(
    base_hamiltonian,
    incompatible_perturbation,
    support,
    parameters,
    reference_state=cage_state,
    tolerance=1.0e-12,
)

branch_table = pd.DataFrame(
    {
        "lambda": parameters,
        "structure_dimension": structure_branch.invariant_dimensions,
        "structure_fixed_residual": structure_branch.fixed_state_full_residuals,
        "structure_continued_residual": structure_branch.continued_full_residuals,
        "incompatible_dimension": incompatible_branch.invariant_dimensions,
    }
)
branch_table.head()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(
    structure_branch.parameters,
    structure_branch.fixed_state_full_residuals,
    marker="o",
    label="unchanged pre-quench state",
)
plt.plot(
    structure_branch.parameters,
    structure_branch.continued_full_residuals,
    marker="s",
    label="continued cage eigenstate",
)
plt.yscale("log")
plt.xlabel(r"deformation $\lambda$")
plt.ylabel("full eigenstate residual")
plt.title("A cage branch can survive while the original state stops being stationary")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
toy_obstruction = linearized_cage_obstruction_from_hamiltonians(
    base_hamiltonian,
    (strong_perturbation, structure_perturbation, incompatible_perturbation),
    support,
    cage_state,
    coefficient_field="real",
    tolerance=1.0e-12,
)

pd.DataFrame(
    [item.to_summary_dict() for item in toy_obstruction.perturbation_diagnostics]
)

In [ ]:
toy_compatible = random_cage_stability_ensemble(
    base_hamiltonian,
    (strong_perturbation, structure_perturbation),
    support,
    strengths=(0.1, 0.5, 1.0),
    n_samples=32,
    reference_state=cage_state,
    target_dimension=1,
    minimum_subspace_overlap=0.5,
    random_seed=7,
    tolerance=1.0e-12,
)
toy_incompatible = random_cage_stability_ensemble(
    base_hamiltonian,
    (incompatible_perturbation,),
    support,
    strengths=(0.1, 0.5, 1.0),
    n_samples=32,
    reference_state=cage_state,
    target_dimension=1,
    minimum_subspace_overlap=0.5,
    random_seed=7,
    tolerance=1.0e-12,
)

pd.DataFrame(
    [
        {"ensemble": "compatible", **item.to_summary_dict()}
        for item in toy_compatible.aggregates
    ]
    + [
        {"ensemble": "incompatible", **item.to_summary_dict()}
        for item in toy_incompatible.aggregates
    ]
)

## 2. Square QDM \(4\times4\), \(W=(0,0)\)

We now use the exact full-basis cage search.  The chosen record belongs to the known \((0,4)\) family.  Its candidate support contains the complete nine-dimensional caged invariant subspace, so the static report distinguishes:

- the boundary-kernel dimension;
- the internally invariant cage dimension;
- the selected record's eigenstate residual and weight inside that invariant subspace.

In [ ]:
square_model = SquareQDMModel(
    lx=4,
    ly=4,
    boundary_condition="periodic",
    winding_x=0,
    winding_y=0,
    winding_convention="electric",
    coup_kin=1.0,
    coup_pot=1.0,
)
square_build = square_model.build(
    basis_solver="dfs",
    builder="sparse",
    backend="scipy",
    sort_basis=True,
)
square_search = CageSearcher.from_model_build_result(
    square_build,
    config=CageSearchConfig(search_type="type1", tolerance=1.0e-10),
).run()

square_record = square_search[(0, 4), 0]
square_baseline = diagnose_cage_stability(
    square_build.hamiltonian,
    square_record.support,
    state=square_record.local_state,
    tolerance=1.0e-10,
)

{
    "hilbert_dimension": square_build.hamiltonian.shape[0],
    "counts_by_signature": square_search.counts_by_signature,
    **square_baseline.to_summary_dict(),
}

### Local perturbation basis and tangent-space codimension

The raw parameter basis consists of all individually built kinetic and potential plaquette operators.  The full tangent obstruction enforces

$$
B\,\delta\phi+\delta B\,\phi=0,
\qquad
(A-E)\delta\phi+(\delta A-\delta E)\phi=0,
\qquad
\langle\phi|\delta\phi\rangle=0.
$$

Its nullspace gives the real coefficient combinations that preserve a caged eigenstate to first order.  This is stronger than testing only $P_{\ker B^\dagger}\delta B|\phi\rangle=0$.

In [ ]:
local_operators = square_build.kinetic_operators + square_build.potential_operators
term_builder = SparseHamiltonianBuilder(
    backend="scipy",
    dtype=np.complex128,
    on_missing="raise",
)
local_term_matrices = tuple(
    term_builder.build(square_build.basis, [operator])
    for operator in local_operators
)

square_obstruction = linearized_cage_obstruction_from_hamiltonians(
    square_build.hamiltonian,
    local_term_matrices,
    square_record.support,
    square_record.local_state,
    coefficient_field="real",
    tolerance=1.0e-10,
)

{
    "n_local_parameters": square_obstruction.n_parameters,
    "obstruction_rank": square_obstruction.rank,
    "compatible_tangent_dimension": square_obstruction.compatible_dimension,
    "compatible_tangent_codimension": (
        square_obstruction.n_parameters - square_obstruction.compatible_dimension
    ),
    "individually_boundary_compatible_terms": sum(
        item.first_order_boundary_compatible
        for item in square_obstruction.perturbation_diagnostics
    ),
    "individually_full_tangent_compatible_terms": sum(
        item.first_order_eigenstate_compatible
        for item in square_obstruction.perturbation_diagnostics
    ),
    "individually_state_preserving_terms": sum(
        item.preserves_state
        for item in square_obstruction.perturbation_diagnostics
    ),
}

The important object is the **coefficient-space nullspace**, not the list of individually compatible terms.  In this benchmark, compatible deformations arise through collective combinations of local operators.

In [ ]:
compatible_term_matrices = combine_perturbations_from_coefficients(
    local_term_matrices,
    square_obstruction.compatible_coefficient_basis,
)
len(compatible_term_matrices)

### Exact finite-amplitude ensemble test

We compare two ensembles:

- random directions inside the full first-order compatible coefficient subspace;
- random directions in the unrestricted raw local-term basis.

The survival test requires an exact caged invariant state and a matched cage eigenstate with overlap at least \(0.5\) with the selected reference record.  The threshold is a continuation diagnostic, not a claimed universal constant.

In [ ]:
compatible_strengths = (1.0e-4, 1.0e-2, 1.0, 10.0)
control_strengths = (1.0e-6, 1.0e-4, 1.0e-2)

square_compatible_ensemble = random_cage_stability_ensemble(
    square_build.hamiltonian,
    compatible_term_matrices,
    square_record.support,
    strengths=compatible_strengths,
    n_samples=3,
    reference_state=square_record.local_state,
    target_dimension=1,
    minimum_subspace_overlap=0.5,
    random_seed=11,
    tolerance=1.0e-9,
)
square_control_ensemble = random_cage_stability_ensemble(
    square_build.hamiltonian,
    local_term_matrices,
    square_record.support,
    strengths=control_strengths,
    n_samples=3,
    reference_state=square_record.local_state,
    target_dimension=1,
    minimum_subspace_overlap=0.5,
    random_seed=11,
    tolerance=1.0e-9,
)

square_ensemble_table = pd.DataFrame(
    [
        {"ensemble": "tangent-compatible", **item.to_summary_dict()}
        for item in square_compatible_ensemble.aggregates
    ]
    + [
        {"ensemble": "unrestricted local control", **item.to_summary_dict()}
        for item in square_control_ensemble.aggregates
    ]
)
square_ensemble_table

In [ ]:
plt.figure(figsize=(7, 4))
for ensemble_name, group in square_ensemble_table.groupby("ensemble"):
    plt.plot(
        group["strength"],
        group["survival_fraction"],
        marker="o",
        label=ensemble_name,
    )
plt.xscale("log")
plt.ylim(-0.05, 1.05)
plt.xlabel("perturbation strength")
plt.ylabel("exact cage survival fraction")
plt.title("Compatible coefficient directions versus generic local directions")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
for ensemble_name, group in square_ensemble_table.groupby("ensemble"):
    plt.plot(
        group["strength"],
        group["minimum_interference_gap"],
        marker="o",
        label=ensemble_name,
    )
plt.xscale("log")
plt.yscale("log")
plt.xlabel("perturbation strength")
plt.ylabel(r"minimum nonzero singular value $\Delta_{\mathrm{I}}$")
plt.title("The interference singular gap detects rank-changing controls")
plt.legend()
plt.tight_layout()
plt.show()

## 3. What the first results mean

For the current archive and the selected square-QDM record, the executed notebook finds:

- a nine-dimensional exact caged invariant subspace on the record's candidate support;
- 48 real local-term parameters;
- an 11-dimensional full first-order compatible coefficient subspace, hence codimension 37;
- no single raw local term that by itself satisfies the full selected-eigenstate tangent condition;
- exact finite-amplitude survival for the sampled collective compatible directions;
- immediate destruction of the exact cage for generic random local directions, even at very small strength.

This is strong evidence for a **collectively defined structurally stable family**.  It is not yet evidence for topology, because we have not shown a discrete invariant or a no-go obstruction to deforming the family into a trivial localized state.

The next numerical stage should therefore compare the suspected robust and fragile cage records using the same tangent codimension, interference-gap statistics, and explicit compatible-path searches.  Only after that comparison should we implement candidate chiral-index and Fock-space homology diagnostics.

## 4. Robust versus fragile records: compatibility hierarchy

The first-order obstruction map is only the tangent equation at the undeformed Hamiltonian.  It can overestimate the set of finite-amplitude cage-preserving paths, especially when the support Hamiltonian has a degenerate eigenspace.

We therefore compare two nested coefficient spaces:

1. the **formal first-order continuation space**, which permits a correction to the cage vector;
2. the **exact fixed-state space**, for which the same compact vector remains an eigenstate of every affine Hamiltonian $H_0+\lambda V$.

The quotient between them consists of tangent-only directions whose finite-amplitude integrability must be tested explicitly.

In [ ]:
robust_record = square_search[(0, 4), 0]
fragile_record = square_search[(0, 6), 0]

robust_hierarchy = cage_compatibility_hierarchy_from_hamiltonians(
    square_build.hamiltonian,
    local_term_matrices,
    robust_record.support,
    robust_record.local_state,
    coefficient_field="real",
    tolerance=1.0e-10,
)
fragile_hierarchy = cage_compatibility_hierarchy_from_hamiltonians(
    square_build.hamiltonian,
    local_term_matrices,
    fragile_record.support,
    fragile_record.local_state,
    coefficient_field="real",
    tolerance=1.0e-10,
)

pd.DataFrame(
    [
        {"record": "robust candidate (0, 4)", **robust_hierarchy.to_summary_dict()},
        {"record": "fragile candidate (0, 6)", **fragile_hierarchy.to_summary_dict()},
    ]
)

The two records have the same 11-dimensional formal tangent space, so tangent codimension alone does not distinguish them.  The finite hierarchy does:

- for the $(0,4)$ cage, all 11 tangent directions already preserve the selected state exactly;
- for the $(0,6)$ cage, only 7 directions are exact, leaving 4 tangent-only directions.

This gives a concrete numerical meaning to “robust” and “fragile” before introducing a topological invariant.

In [ ]:
robust_fixed_directions = combine_perturbations_from_coefficients(
    local_term_matrices,
    robust_hierarchy.fixed_state.compatible_coefficient_basis,
)
fragile_fixed_directions = combine_perturbations_from_coefficients(
    local_term_matrices,
    fragile_hierarchy.fixed_state.compatible_coefficient_basis,
)
fragile_tangent_directions = combine_perturbations_from_coefficients(
    local_term_matrices,
    fragile_hierarchy.first_order.compatible_coefficient_basis,
)
fragile_tangent_only_directions = combine_perturbations_from_coefficients(
    local_term_matrices,
    fragile_hierarchy.tangent_only_coefficient_basis,
)

{
    "robust_exact_directions": len(robust_fixed_directions),
    "fragile_exact_directions": len(fragile_fixed_directions),
    "fragile_formal_tangent_directions": len(fragile_tangent_directions),
    "fragile_tangent_only_directions": len(fragile_tangent_only_directions),
}

### Finite-amplitude integrability of a tangent-only direction

For a direction that satisfies the formal first-order equation but is not exactly state-preserving, we continue the closest eigenspace of the internal support Hamiltonian.  Inside each degenerate eigenspace, the algorithm chooses the state with minimum boundary leakage.  This avoids mistaking an arbitrary basis choice inside a degenerate level for the physical near-cage branch.

In [ ]:
path_parameters = np.concatenate(([0.0], np.logspace(-5, -2, 7)))

robust_fixed_branch = scan_support_eigenstate_branch(
    square_build.hamiltonian,
    robust_fixed_directions[0],
    robust_record.support,
    path_parameters,
    reference_state=robust_record.local_state,
    tolerance=1.0e-11,
)
fragile_fixed_branch = scan_support_eigenstate_branch(
    square_build.hamiltonian,
    fragile_fixed_directions[0],
    fragile_record.support,
    path_parameters,
    reference_state=fragile_record.local_state,
    tolerance=1.0e-11,
)
fragile_tangent_only_branch = scan_support_eigenstate_branch(
    square_build.hamiltonian,
    fragile_tangent_only_directions[0],
    fragile_record.support,
    path_parameters,
    reference_state=fragile_record.local_state,
    tolerance=1.0e-11,
)

fragile_leakage_exponent = estimate_power_law_exponent(
    fragile_tangent_only_branch.parameters,
    fragile_tangent_only_branch.boundary_residuals,
    minimum_parameter=1.0e-6,
    minimum_residual=1.0e-13,
)

pd.DataFrame(
    {
        "lambda": path_parameters,
        "robust_exact_residual": robust_fixed_branch.boundary_residuals,
        "fragile_exact_residual": fragile_fixed_branch.boundary_residuals,
        "fragile_tangent_only_residual": (
            fragile_tangent_only_branch.boundary_residuals
        ),
        "fragile_tangent_only_cage_dimension": [
            point.invariant_cage_dimension
            for point in fragile_tangent_only_branch.points
        ],
    }
)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(
    path_parameters[1:],
    np.maximum(robust_fixed_branch.boundary_residuals[1:], 1.0e-16),
    marker="o",
    label="(0, 4): exact compatible direction",
)
plt.plot(
    path_parameters[1:],
    np.maximum(fragile_fixed_branch.boundary_residuals[1:], 1.0e-16),
    marker="s",
    label="(0, 6): exact compatible direction",
)
plt.plot(
    path_parameters[1:],
    fragile_tangent_only_branch.boundary_residuals[1:],
    marker="^",
    label="(0, 6): tangent-only direction",
)
plt.xscale("log")
plt.yscale("log")
plt.xlabel(r"deformation $|\lambda|$")
plt.ylabel(r"minimum support-eigenstate leakage $\|B\phi\|$")
plt.title("A formal tangent direction develops finite-amplitude leakage")
plt.legend()
plt.tight_layout()
plt.show()

{
    "fragile_tangent_only_leakage_exponent": fragile_leakage_exponent,
    "nonzero_path_cage_dimensions": tuple(
        point.invariant_cage_dimension
        for point in fragile_tangent_only_branch.points[1:]
    ),
}

The tangent-only leakage scales as approximately $\lambda^2$, while the exact directions remain at numerical precision.  Thus the fragile record satisfies the linearized solvability condition but immediately loses its exact cage at every nonzero deformation on this path.  This is precisely the failure that a quench of only hand-selected fixed-state-compatible terms cannot reveal.

In [ ]:
fragile_tangent_ensemble = random_cage_stability_ensemble(
    square_build.hamiltonian,
    fragile_tangent_directions,
    fragile_record.support,
    strengths=(1.0e-2, 1.0),
    n_samples=3,
    reference_state=fragile_record.local_state,
    target_dimension=1,
    minimum_subspace_overlap=0.5,
    random_seed=23,
    tolerance=1.0e-9,
)
fragile_fixed_ensemble = random_cage_stability_ensemble(
    square_build.hamiltonian,
    fragile_fixed_directions,
    fragile_record.support,
    strengths=(1.0e-2, 1.0),
    n_samples=3,
    reference_state=fragile_record.local_state,
    target_dimension=1,
    minimum_subspace_overlap=0.5,
    random_seed=23,
    tolerance=1.0e-9,
)

pd.DataFrame(
    [
        {"ensemble": "formal tangent", **item.to_summary_dict()}
        for item in fragile_tangent_ensemble.aggregates
    ]
    + [
        {"ensemble": "exact fixed-state", **item.to_summary_dict()}
        for item in fragile_fixed_ensemble.aggregates
    ]
)

## 5. Current conclusion and next invariant test

The comparison now supplies a reproducible finite-size distinction:

- **robust $(0,4)$ record:** formal tangent space = exact affine compatibility space;
- **fragile $(0,6)$ record:** formal tangent space strictly contains the exact space, and the extra directions acquire quadratic leakage.

This still does not prove topology.  The next diagnostic should explain *why* the $(0,4)$ compatibility equations close exactly.  The most targeted next step is to construct the support–boundary interference network, compute its structured row dependencies and cycle/holonomy data, and test which of those quantities remains unchanged across the 11-dimensional exact family but fails for the four non-integrable directions.